# Avance 4 — Modelos alternativos (Equipo 17, AgroSatCopilot)

## Proyecto Integrador MNA · Tec de Monterrey

**Equipo 17**

- Carlos Isaac Ávila Gutiérrez — A01796035
- Carlos Aaron Bocanegra Buitrón — A01796345
- Arthur Jafed Zizumbo Velasco — A01796363

**Curso**: MNA — Tec de Monterrey · 20-abr → 3-jul-2026

**Sponsor académico**: Dr. Gerardo José Camacho — gjcamacho@tec.mx

**Fecha de entrega**: 2026-05-31.

---

## Resumen ejecutivo

Este cuaderno **consolida** las **6 arquitecturas individuales** (no ensambles) de segmentación semántica densa de cultivos sobre PASTIS-R, repartidas entre el equipo. Consume los parquets comparativos que cada integrante exporta, construye la tabla de los 6 modelos ordenada por la métrica principal (mIoU), selecciona el top-2, documenta el ajuste fino (Optuna) y la elección del modelo individual final.

| # | Modelo | Tipo |
|---|--------|------|
| 1 | U-Net ResNet-50 | CNN clásica 2D |
| 2 | DeepLabv3+ MobileNet | CNN eficiente ASPP |
| 3 | SegFormer-B0 | Transformer spatial |
| 4 | U-TAE | Temporal Attention |
| 5 | TSViT | Transformer temporal (Paper 1) |
| 6 | AnySat frozen + linear head | Foundation model congelado |

> Cada integrante entrena sus modelos con `ml.train.train_segmentation.run_training` y exporta su parquet a `reports/segmentation/model_comparison_avance4_<nombre>.parquet`. Este notebook los une.

## Objetivos y rubrica

- **3.3** Explorar una gama diversa de tecnicas (6 arquitecturas).
- **3.4** Encontrar la configuracion optima (ajuste fino del top-2).

**Distribucion de puntos**:

- **Comparativa (60 pts)**: >=6 modelos, tabla ordenada por mIoU + F1-macro + pixel-accuracy + tiempos de entrenamiento.
- **Ajuste fino (30 pts)**: Optuna (>=30 trials) sobre los 2 mejores.
- **Modelo individual (10 pts)**: justificacion del final (trade-offs, no solo metrica).

In [ ]:
# --- Setup Colab + Drive compartido del equipo ---
import os, sys
from pathlib import Path

# Mismo patron que 04_baseline: monta Drive y prefija las rutas con
# shared_folder_path (vacio en local).
_IN_COLAB = False
shared_folder_path = ''
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shared_folder_path = '/content/drive/MyDrive/Integrador/'
    _IN_COLAB = True
except ImportError:
    pass

_search = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if _IN_COLAB:
    _search = [Path('/content/agrosat-copilot'), *_search]
for _cand in _search:
    if (_cand / 'pyproject.toml').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        os.chdir(_cand)
        break
else:
    raise RuntimeError('No se encontro el repo agrosat-copilot (pyproject.toml).')

import matplotlib.pyplot as plt
import polars as pl
print('repo:', Path.cwd(), '| colab:', _IN_COLAB, '| drive:', shared_folder_path or '(local)')

## Metodologia

- **Dataset**: PASTIS-R (2433 patches Sentinel-2 multitemporales, 20 clases, folds oficiales espacialmente disjuntos -> sin leakage).
- **Convencion compartida**: `num_classes=20`, `ignore_index=19` (void), resolucion **256px**, split train=folds[1,2,3] / val=fold[4] / test=fold[5].
- **Metricas**: mIoU (principal) + F1-macro + pixel-accuracy, todas pixel-level (`ml/eval/dense_metrics.py`).
- **Tracking**: 1 run MLflow por modelo con tag `architecture`.

## Comparativa de los 6 modelos (60 pts)

In [ ]:
# --- Consolidar los parquets de los integrantes (desde el Drive compartido) ---
REPORTS = Path((shared_folder_path if shared_folder_path else '')
               + 'reports/segmentation/metrics')
parts = sorted(REPORTS.glob('model_comparison_avance4_*.parquet'))
print('parquets encontrados:', [p.name for p in parts])

if parts:
    table = pl.concat([pl.read_parquet(p) for p in parts], how='vertical_relaxed')
    table = table.unique(subset=['model'], keep='last').sort('miou', descending=True)
else:
    print('Aun no hay parquets. Corre primero 04d_segmentation_unet_anysat.ipynb '
          'y los notebooks de los demas modelos.')
    table = pl.DataFrame()

_cols = ['model', 'miou_grouped', 'f1_macro_grouped', 'pixel_accuracy_grouped',
         'miou', 'f1_macro', 'pixel_accuracy', 'train_time_s', 'epochs']
table.select([c for c in _cols if c in table.columns]) if table.height else table

In [ ]:
# --- Barplot comparativo (mIoU por modelo) ---
if table.height:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(table['model'].to_list()[::-1], table['miou'].to_list()[::-1], color='#2b6cb0')
    ax.set_xlabel('mIoU (val)')
    ax.set_title('Avance 4 - Comparativa de arquitecturas de segmentacion')
    fig.tight_layout()
    display(fig)
else:
    print('Tabla vacia: nada que graficar todavia.')

## Seleccion de los 2 mejores modelos

Por la metrica principal (mIoU); empate se rompe por F1-macro -> pixel-accuracy.

In [ ]:
# --- Top-2 ---
if table.height >= 2:
    top2 = table.head(2)
    print('Top-2 por mIoU:', top2['model'].to_list())
    top2.select([c for c in _cols if c in top2.columns])
else:
    print('Se requieren >=2 modelos para seleccionar el top-2.')
    top2 = table

## Ajuste fino del top-2 (30 pts)

Cada modelo del top-2 se afina con **Optuna (>=30 trials)** sobre `lr`, `weight_decay` y `batch_size`, reusando `ml.train.train_segmentation.run_training` (ver hook en `04d_segmentation_unet_anysat.ipynb`). Los resultados afinados se exportan a `reports/segmentation/tuning_<modelo>.parquet` y se cargan aqui.

In [ ]:
# --- Resultados del ajuste fino (si existen) ---
tuning_parts = sorted(REPORTS.glob('tuning_*.parquet'))
if tuning_parts:
    tuning = pl.concat([pl.read_parquet(p) for p in tuning_parts], how='vertical_relaxed')
    display(tuning)
else:
    print('Pendiente: ejecutar Optuna sobre el top-2 una vez confirmados los 6 modelos.')
    tuning = pl.DataFrame()

## Modelo individual final (10 pts)

Eleccion argumentada con trade-offs (no solo la metrica):

- **Rendimiento**: mIoU / F1-macro tras el ajuste fino.
- **Costo de computo**: tiempo de entrenamiento e inferencia (relevante para el presupuesto L4/H100 del proyecto).
- **Interpretabilidad y robustez**: las CNN (U-Net/DeepLabv3+) son mas simples de diagnosticar; los modelos temporales (U-TAE/TSViT) capturan fenologia; AnySat ofrece un FM congelado con minima capacidad entrenable.

> Completar tras la corrida real con el modelo ganador y su justificacion.

In [ ]:
# --- Modelo final + matriz de confusion ---
if table.height:
    final_model = table.row(0, named=True)['model']
    print('Modelo individual final (preliminar por mIoU):', final_model)
    # La matriz de confusion del modelo final se genera en el notebook del
    # responsable (p.ej. 04d para unet/anysat) con dense_confusion_figure.
else:
    print('Definir el modelo final tras consolidar los 6.')

## Conclusiones y checklist de la rubrica

- [ ] **Comparativa (60)**: >=6 modelos en la tabla, ordenados por mIoU + F1-macro + pixel-accuracy + tiempos.
- [ ] **Ajuste fino (30)**: Optuna >=30 trials sobre el top-2, mejora documentada.
- [ ] **Modelo individual (10)**: final elegido con argumentos de trade-offs.

**Entrega**: liga de GitHub, ejecucion secuencial, nombre `Avance4.Equipo17`.